# NetKet Benchmark

This notebook is the user-facing comparison notebook. It runs TFIM and J1-J2 on the project workflow for RBM, FFNN, and CNN, then places those results next to exact or NetKet reference energies. JAX `x64` is enabled only inside this notebook session.

In [ ]:
from notebook_bootstrap import bootstrap_notebook

bootstrap_notebook(enable_x64=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from netket_reference import (
    exact_netket_j1j2_ground_energy,
    exact_netket_tfim_ground_energy,
)
from nqs.workflows import run_architecture_benchmark


In [ ]:
architecture_configs = {
    "RBM": {"alpha": 1},
    "FFNN": {"hidden_dims": (8,)},
    "CNN": {"channels": (4,), "kernel_size": (2, 2)},
}

benchmark_config = {
    "lattice_shape": (2, 2),
    "pbc": False,
    "n_iter": 8,
    "n_samples": 64,
    "n_discard_per_chain": 8,
    "n_chains": 8,
    "callback_every": 1,
}

## TFIM Benchmark

In [ ]:
tfim_reference_energy = exact_netket_tfim_ground_energy(
    lattice_shape=benchmark_config["lattice_shape"],
    pbc=benchmark_config["pbc"],
    J=1.0,
    h=1.0,
)
tfim_benchmark = run_architecture_benchmark(
    architecture_configs=architecture_configs,
    hamiltonian="tfim",
    J=1.0,
    h=1.0,
    netket_reference_energy=tfim_reference_energy,
    **benchmark_config,
)
tfim_benchmark["summary_table"]

## J1-J2 Benchmark

In [ ]:
j1j2_reference_energy = exact_netket_j1j2_ground_energy(
    lattice_shape=benchmark_config["lattice_shape"],
    pbc=benchmark_config["pbc"],
    J1=1.0,
    J2=0.5,
)
j1j2_benchmark = run_architecture_benchmark(
    architecture_configs=architecture_configs,
    hamiltonian="j1_j2",
    J1=1.0,
    J2=0.5,
    netket_reference_energy=j1j2_reference_energy,
    **benchmark_config,
)
j1j2_benchmark["summary_table"]

## Combined Comparison Table

In [ ]:
combined_summary = pd.concat(
    [
        tfim_benchmark["summary_table"].assign(case="TFIM"),
        j1j2_benchmark["summary_table"].assign(case="J1-J2"),
    ],
    ignore_index=True,
)[
    ["case", "model", "final_energy", "exact_ground_energy", "energy_error", "netket_reference_energy", "netket_gap", "final_renyi2_entropy"]
]
combined_summary

## Visual Comparison

In [ ]:
plot_table = combined_summary.copy()
plot_table["label"] = plot_table["case"] + " / " + plot_table["model"]

figure, axis = plt.subplots(figsize=(10, 4.5))
axis.bar(plot_table["label"], plot_table["energy_error"], color="#2B6CB0")
axis.axhline(0.0, color="black", linewidth=1.0)
axis.set_ylabel("Project energy minus exact energy")
axis.set_title("Architecture benchmark error by model family")
axis.tick_params(axis="x", rotation=35)
axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
figure